In [ ]:
import re
from collections import Counter

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import CountVectorizer, ENGLISH_STOP_WORDS

In [ ]:
data_clean = "goodreads_onestar_clean.csv"   # cleaned data, one-star reviews only
TEXT_COL = "review_text"   # name of the column with the review text
MIN_WORDS = 10             # skip reviews shorter than this many words
SEED = 42 # random seed for reproducibility

HIDE = ["alice", "wonderland", "carroll", "silverstein", "peter"] # common titles and authors to hide

In [ ]:
one_star = pd.read_csv(data_clean) # turning into a DataFrame
one_star = one_star.dropna(subset=[TEXT_COL]) # dropping reviews with no text
one_star = one_star[one_star[TEXT_COL].str.split().str.len() >= MIN_WORDS] # splitting review text into list of words, counting them, and keeping only reviews with at least MIN_WORDS words

print(f"1-star reviews: {len(one_star)}") # printing the number of 1-star reviews after filtering

### Flagging reviews about "appropriateness"

In [ ]:
def compare_groups(texts_a, texts_b):  # scoring words and phrases that are distinctive to one group vs another
    texts_a = list(texts_a) # a positive score = more typical of group A, negative score = more typical of group B
    texts_b = list(texts_b)

    counter = CountVectorizer( # using CountVectorizer to count words and phrases in the two groups
        ngram_range=(1, 2), # defining upper and lower limit of words to be extracted
        min_df=15,    # words/phrases must appear in 15+ reviews
        token_pattern=r"(?u)\b\w[\w']*\b",   # keeps negations and other words with apostrophes like "don't" 
    )
    counts = counter.fit_transform(texts_a + texts_b) # makes table of counts of each word/phrase in each review
        
    a_review_number = len(texts_a) # how many reviews in group A and group B
    b_review_number = len(texts_b)
    count_a = np.asarray(counts[:a_review_number].sum(axis=0)).ravel() # selects group A reviews, sums the counts of each term across all reviews, and flattens into list of numbers
    count_b = np.asarray(counts[a_review_number:].sum(axis=0)).ravel() # selects group B reviews, sums the counts of each term across all reviews, and flattens into list of numbers

    baseline = count_a + count_b # how many times each term appears in all reviews (group A + group B)
    baseline_total = baseline.sum() # sums across all terms to get the total number of term appearances in all reviews

    odds_a = np.log((count_a + baseline) / (count_a.sum() + baseline_total - count_a - baseline)) # computes the log-odds of each term appearing in group A vs group B, with a baseline to avoid division by zero
    odds_b = np.log((count_b + baseline) / (count_b.sum() + baseline_total - count_b - baseline))
    difference = odds_a - odds_b

    # Turn the difference into a score that accounts for how much data there is
    uncertainty = np.sqrt(1 / (count_a + baseline) + 1 / (count_b + baseline)) # this calculates uncertainty for each term based on its counts in both groups and the baseline 
    score = difference / uncertainty # dividing to get a z-score-like measure

    results = pd.DataFrame({ # putting everything into a DataFrame 
        "term": counter.get_feature_names_out(),
        "score": score,
        "count_A": count_a,
        "count_B": count_b,
    })

        
    for name in HIDE: # hiding common titles and authors from the results
        results = results[~results["term"].str.contains(r"\b" + name + r"\b")]

    return results.sort_values("score", ascending=False).reset_index(drop=True) # sorting the results by score in descending order and resetting the index


In [ ]:
# These categories describe why a review entered the candidate set. They are
# screening aids only; the final appropriateness aspect will be hand labeled.
TRIGGER_PATTERNS = {
    "explicit_appropriateness": (
        r"\b(?:age[- ]?)?inappropriate\w*\b"
        r"|\b(?:not|isn't|wasn't|aren't|weren't)\s+(?:\w+\s+){0,2}(?:age[- ]?)?appropriate\b"
        r"|\bnot\s+suitable\b|\bunsuitable\b"
    ),
    "offensive_or_moral_language": r"\b(?:offensive|vulgar|crude|perverse|immoral|sinful)\b",
    "content_intensity": r"\btoo\s+(?:scary|violent|graphic|dark|mature|disturbing|sexual|adult|intense|creepy)\b",
    "age_mismatch": r"\btoo\s+(?:old|young|mature|advanced)\s+for\b",
    "audience_mismatch": (
        r"\bnot\s+(?:for|meant\s+for|intended\s+for)\s+(?:\w+\s+){0,2}(?:kids|children|young)\b"
        r"|\bnot\s+(?:very\s+)?(?:child|kid|family)[- ]friendly\b"
    ),
}

# Use a non-capturing pattern for screening and named groups for match details.
TRIGGER_PATTERN = "|".join(f"(?:{pattern})" for pattern in TRIGGER_PATTERNS.values())
LABELED_TRIGGER_PATTERN = "|".join(
    f"(?P<{name}>{pattern})" for name, pattern in TRIGGER_PATTERNS.items()
)
WINDOW = 12

In [ ]:
has_trigger = one_star[TEXT_COL].str.contains(TRIGGER_PATTERN, case=False, regex=True) # checking which reviews contain the trigger pattern, ignoring case and using regex
matched = one_star[has_trigger]
others = one_star[~has_trigger]
print(f"{len(matched):,} of {len(one_star):,} 1-star reviews contain an " # printing the number and percentage of 1-star reviews that contain an inappropriateness complaint
      f"inappropriateness complaint ({len(matched) / len(one_star):.1%})")

In [ ]:
def first_match_details(text, width=WINDOW):
    m = re.search(LABELED_TRIGGER_PATTERN, text, flags=re.I)
    before = " ".join(text[:m.start()].split()[-width:])
    after = " ".join(text[m.end():].split()[:width])
    return pd.Series({
        "matched_trigger": m.group(0),
        "trigger_family": m.lastgroup,
        "snippet": f"{before} [[{m.group(0)}]] {after}",
    })


def all_triggers(text):
    # Preserve order while removing duplicate matched phrases.
    matches = (m.group(0) for m in re.finditer(LABELED_TRIGGER_PATTERN, text, flags=re.I))
    return " | ".join(dict.fromkeys(matches))


matched = matched.copy()
matched[["matched_trigger", "trigger_family", "snippet"]] = matched[TEXT_COL].apply(first_match_details)
matched["all_triggers"] = matched[TEXT_COL].apply(all_triggers)

pd.set_option("display.max_colwidth", None) # letting the entire snippet show up in the output, instead of being truncated
print("\n25 random examples (read these first):")
print(matched[["book_id", "trigger_family", "snippet"]].sample(min(25, len(matched)), # printing 25 random examples of the matched reviews, showing the book_id and the snippet, and using a random seed for reproducibility
                                             random_state=SEED).to_string())

In [ ]:
distinct = compare_groups(matched[TEXT_COL], others[TEXT_COL]) # running the compare_groups function on the matched and others DataFrames to get the most distinctive words in the matched reviews
distinct = distinct[~distinct["term"].str.contains("appropriat|suitab|offen")] # removing the trigger words from the results, so that the most distinctive words are not just the trigger words themselves
distinct.to_csv("words_inappropriate_vs_other_1star.csv", index=False)
print("\nMost distinctive words in reviews that call something inappropriate:")
print(distinct.head(40).round(2).to_string())

#### CSV containing all candidate reviews, screening information, and blank HAND LABEL columns

In [ ]:
# Export every candidate rather than dropping reviews through random sampling.
# The original row index is retained as a stable ID when no review_id is available.
matched = matched.rename_axis("source_row_id").reset_index()
id_columns = [column for column in ["review_id", "source_row_id", "book_id"] if column in matched.columns]
screening_columns = ["matched_trigger", "trigger_family", "all_triggers", "snippet"]
to_code = matched[id_columns + [TEXT_COL] + screening_columns].copy()

# These fields are intentionally blank for manual labeling.
to_code["is_inappropriate_complaint"] = ""
to_code["aspect"] = ""
to_code["notes"] = ""

to_code.to_csv("inappropriate_to_code.csv", index=False, encoding="utf-8-sig")
print("\nSaved inappropriate_to_code.csv with", len(to_code), "reviews to hand label")

### Looking at NOT-flagged reviews (reviews not about "appropriateness")

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.decomposition import NMF

print("\nWords most typical of the NOT-flagged reviews:")
print(distinct.sort_values("score").head(40).round(2).to_string(index=False)) # printing the 40 words with the lowest score, which are the most typical of the NOT-flagged reviews


In [ ]:
print("\n15 random NOT-flagged reviews (first 300 characters):")
for text in others[TEXT_COL].sample(15, random_state=SEED): # a loop taking a random sample of 15 NOT-flagged reviews
    print("-", text[:300].replace("\n", " "), "\n")

#### Topic Model of NOT-flagged reviews

In [ ]:
NUMBER_OF_TOPICS = 8
tfidf = TfidfVectorizer(stop_words="english", min_df=10, max_df=0.5) # using TF-IDF to assign weights to words in the reviews
X = tfidf.fit_transform(others[TEXT_COL]) # transforming the NOT-flagged reviews into a matrix
nmf = NMF(n_components=NUMBER_OF_TOPICS, random_state=SEED, init="nndsvda", max_iter=400) # using NMF to find topics in the NOT-flagged reviews, with 8 topics, a random seed for reproducibility, and a maximum of 400 iterations
doc_topics = nmf.fit_transform(X) # transforming the matrix of NOT-flagged reviews into a matrix of topic weights for each review
terms = tfidf.get_feature_names_out() # getting the feature names (words) from the TF-IDF vectorizer

others = others.copy()
others["topic"] = doc_topics.argmax(axis=1) # assigning each review to the topic with the highest weight
for i, weights in enumerate(nmf.components_): # a loop that goes through each topic and prints the top 10 words for that topic, along with the number of reviews assigned to that topic
    top_words = [terms[j] for j in weights.argsort()[::-1][:10]]
    n_reviews = (others["topic"] == i).sum()
    print(f"\nTopic {i} ({n_reviews:,} reviews): {', '.join(top_words)}")